In [20]:
import pandas as pd
import matplotlib.pyplot as plt

In [17]:
platforms_df = pd.read_csv('metrika-platforms.csv')
totals = platforms_df.iloc[0]
platforms_df = platforms_df.drop(index=0)

platforms_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 1 to 248
Data columns (total 10 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   UTM Content                                             248 non-null    object 
 1   Площадка                                                248 non-null    object 
 2   Визиты                                                  248 non-null    int64  
 3   Посетители                                              248 non-null    int64  
 4   Отказы                                                  248 non-null    float64
 5   Время на сайте                                          248 non-null    object 
 6   Роботность                                              248 non-null    float64
 7   Достижения цели (Автоцель: заполнил контактные данные)  248 non-null    int64  
 8   Достижения цели (click_download)        

In [27]:
unique_utm = platforms_df['UTM Content'].unique()

# Loop over each unique UTM Content
for utm in unique_utm:
    # Filter rows for the current UTM Content
    subset = platforms_df[platforms_df['UTM Content'] == utm]
    
    # Calculate total visits for this UTM content
    total_visits = subset['Визиты'].sum()
    
    # Skip if total visits is zero to avoid division issues
    if total_visits == 0:
        print(f"Skipping {utm}: total visits = 0")
        continue
    
    # Step 1: Find platforms that contribute >= 10% of total visits
    platform_visits = subset.groupby('Площадка')['Визиты'].sum()
    min_visits_threshold = total_visits * 0.1
    qualifying_platforms = platform_visits[platform_visits >= min_visits_threshold].index
    
    # Step 2: Filter the subset to only those qualifying platforms
    filtered_subset = subset[subset['Площадка'].isin(qualifying_platforms)]
    
    # Step 3: Aggregate 'Отказы' on the filtered data
    platform_bounces = (
        filtered_subset.groupby('Площадка')['Отказы']
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    
    # Print result
    if not platform_bounces.empty:
        print(f"--- UTM Content: {utm} (Total visits: {total_visits}) ---")
        print(platform_bounces)
        print()
    else:
        print(f"--- UTM Content: {utm} --- No platform meets the 10% visit threshold.")
        print()

--- UTM Content: K5_welcome-bonus (Total visits: 584) ---
Площадка
dsp.yandex.ru    0.44964
Name: Отказы, dtype: float64

--- UTM Content: K1_compare-save (Total visits: 20) ---
Площадка
dsp-opera-exchange.yandex.ru    0.500000
dsp.yandex.ru                   0.333333
Не определено                   0.333333
Name: Отказы, dtype: float64

--- UTM Content: K3_no-activation (Total visits: 1117) ---
Площадка
dsp.yandex.ru                   0.491525
dsp-opera-exchange.yandex.ru    0.472050
Name: Отказы, dtype: float64

--- UTM Content: K2_cashback-on-site (Total visits: 452) ---
Площадка
dsp.yandex.ru    0.451852
Name: Отказы, dtype: float64

--- UTM Content: K4_seller-check (Total visits: 9) ---
Площадка
dsp.yandex.ru                   0.666667
dsp-opera-exchange.yandex.ru    0.500000
Не определено                   0.500000
com.tripledot.woodoku           0.000000
dsp-yeahmobi.yandex.ru          0.000000
Name: Отказы, dtype: float64

--- UTM Content: r9_ip-HbqZgSdpBJ (Total visits: 2) ---

In [32]:
import pandas as pd

# Define the two achievement columns
col_contact = 'Достижения цели (Автоцель: заполнил контактные данные)'
col_download = 'Достижения цели (click_download_header)'

# Fill missing values with 0 to avoid errors during aggregation
platforms_df[col_contact] = platforms_df[col_contact].fillna(0)
platforms_df[col_download] = platforms_df[col_download].fillna(0)

# =======================================================
# 1. GLOBAL STATISTICS (across all UTM Contents)
# =======================================================
global_stats = platforms_df.groupby('Площадка')[[col_contact, col_download]].sum()

# Top 10 platforms for the "Contact Data" achievement
top10_contact_global = global_stats[col_contact].sort_values(ascending=False).head(10)

# Top 10 platforms for the "Download Header" achievement
top10_download_global = global_stats[col_download].sort_values(ascending=False).head(10)

print("=" * 60)
print("GLOBAL STATISTICS")
print("=" * 60)
print("\nTop 10 platforms with most 'fill in contact data':")
print(top10_contact_global)
print("\nTop 10 platforms with most 'click_download_header':")
print(top10_download_global)

# =======================================================
# 2. PER‑UTM STATISTICS (broken down by UTM Content)
# =======================================================
unique_utms = platforms_df['UTM Content'].unique()

print("\n" + "=" * 60)
print("PER‑UTM STATISTICS")
print("=" * 60)

for utm in unique_utms:
    # Filter to the current UTM Content
    subset = platforms_df[platforms_df['UTM Content'] == utm]
    
    # Group by platform and sum both achievement columns
    utm_stats = subset.groupby('Площадка')[[col_contact, col_download]].sum()
    
    # Top 10 for the "Contact Data" achievement within this UTM
    top_contact = utm_stats[col_contact].sort_values(ascending=False).head(10)
    
    # Top 10 for the "Download Header" achievement within this UTM
    top_download = utm_stats[col_download].sort_values(ascending=False).head(10)
    
    print(f"\n--- UTM Content: {utm} ---")
    print(f"Total visits for this UTM: {subset['Визиты'].sum()}")
    
    print(f"\nTop 10 platforms for '{col_contact}':")
    if not top_contact.empty:
        print(top_contact)
    else:
        print("  (No data)")
    
    print(f"\nTop 10 platforms for '{col_download}':")
    if not top_download.empty:
        print(top_download)
    else:
        print("  (No data)")

GLOBAL STATISTICS

Top 10 platforms with most 'fill in contact data':
Площадка
Не определено                   3
ya.ru                           1
com.words.puzzle.word.search    0
com.tyou.cutiekitty.xm          0
com.tyou.paradise.mi            0
com.tzl.campus.xm               0
com.tzl.crafts.mi               0
com.uc.sprunki.transform        0
com.vesna.solitaire.klondike    0
com.vitastudio.mahjong          0
Name: Достижения цели (Автоцель: заполнил контактные данные), dtype: int64

Top 10 platforms with most 'click_download_header':
Площадка
Не определено                   6
ya.ru                           2
dsp.yandex.ru                   2
com.words.puzzle.word.search    0
com.tyou.paradise.mi            0
com.tzl.campus.xm               0
com.tzl.crafts.mi               0
com.uc.sprunki.transform        0
com.vesna.solitaire.klondike    0
com.vitastudio.mahjong          0
Name: Достижения цели (click_download_header), dtype: int64

PER‑UTM STATISTICS

--- UTM Content: K5_wel